# Faultline AMD compute setup

Sets up Qwen3-4B on the existing ROCm PyTorch runtime. It does not modify PyTorch, the Spark, or Netplan. The model server listens on loopback and requires an API key. Run each cell in order; wait for dependency installation to complete before starting the server. The final public endpoint step is disabled until you explicitly enable it.


In [ ]:
import sys, subprocess, os, secrets, json
from pathlib import Path
setup_dir = Path('/workspace/faultline-amd')
setup_dir.mkdir(exist_ok=True)
setup_log = (setup_dir / 'install.log').open('w')
setup_process = subprocess.Popen([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.1', 'accelerate==1.10.1', 'fastapi==0.119.0', 'uvicorn==0.37.0'], stdout=setup_log, stderr=subprocess.STDOUT)
print('Dependency installer started:', setup_process.pid)


In [ ]:
print('Installer exit code:', setup_process.poll())
print((setup_dir / 'install.log').read_text()[-2000:])


In [ ]:
if setup_process.poll() != 0:
    raise RuntimeError('Dependency installation must finish successfully first')
server_source = 'import asyncio\nimport json\nimport os\nimport secrets\nimport time\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\n\nfrom fastapi import Depends, FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nMODEL = os.environ.get("AMD_MODEL", "Qwen/Qwen3-4B")\nKEY_FILE = Path(os.environ.get("AMD_KEY_FILE", "/workspace/faultline-amd/api-key"))\nstate = {"ready": False, "active": False}\n\n\ndef load_model():\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    if not torch.cuda.is_available() or not torch.version.hip:\n        raise RuntimeError("An AMD ROCm GPU is required")\n    state["tokenizer"] = AutoTokenizer.from_pretrained(MODEL)\n    state["model"] = AutoModelForCausalLM.from_pretrained(\n        MODEL,\n        torch_dtype=torch.bfloat16,\n        device_map="cuda:0",\n        attn_implementation="sdpa",\n    ).eval()\n    state["gpu"] = torch.cuda.get_device_name(0)\n    state["rocm"] = torch.version.hip\n    state["ready"] = True\n\n\n@asynccontextmanager\nasync def lifespan(application):\n    state["key"] = KEY_FILE.read_text().strip()\n    if len(state["key"]) < 32:\n        raise RuntimeError("A strong API key is required")\n    await asyncio.to_thread(load_model)\n    yield\n\n\napp = FastAPI(docs_url=None, redoc_url=None, openapi_url=None, lifespan=lifespan)\n\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(\n        authorization, f"Bearer {state.get(\'key\', \'\')}"\n    ) or not state.get("key"):\n        raise HTTPException(401, "Unauthorized")\n\n\nclass Message(BaseModel):\n    role: str\n    content: str = Field(max_length=100000)\n\n\nclass CompletionRequest(BaseModel):\n    model: str\n    messages: list[Message] = Field(min_length=1, max_length=32)\n    max_tokens: int = Field(default=256, ge=1, le=512)\n    temperature: float = Field(default=0, ge=0, le=2)\n    stream: bool = False\n    response_format: dict | None = None\n\n\n@app.get("/health", dependencies=[Depends(authorize)])\ndef health():\n    return {\n        "ready": state["ready"],\n        "model": MODEL,\n        "gpu": state.get("gpu"),\n        "rocm": state.get("rocm"),\n        "active_requests": int(state["active"]),\n        "capacity": 1,\n    }\n\n\n@app.get("/v1/models", dependencies=[Depends(authorize)])\ndef models():\n    return {\n        "object": "list",\n        "data": [{"id": MODEL, "object": "model", "owned_by": "faultline-amd"}],\n    }\n\n\ndef generate(request):\n    import torch\n\n    tokenizer = state["tokenizer"]\n    model = state["model"]\n    prompt = tokenizer.apply_chat_template(\n        [message.model_dump() for message in request.messages],\n        tokenize=False,\n        add_generation_prompt=True,\n        enable_thinking=False,\n    )\n    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)\n    input_tokens = inputs.input_ids.shape[1]\n    if input_tokens > 8192:\n        raise HTTPException(413, "Prompt exceeds 8192 tokens")\n    with torch.inference_mode():\n        output = model.generate(\n            **inputs,\n            max_new_tokens=request.max_tokens,\n            do_sample=False,\n            pad_token_id=tokenizer.eos_token_id,\n        )\n    content = tokenizer.decode(\n        output[0][input_tokens:], skip_special_tokens=True\n    ).strip()\n    if request.response_format and request.response_format.get("type") == "json_object":\n        try:\n            if not isinstance(json.loads(content), dict):\n                raise ValueError()\n        except ValueError:\n            raise HTTPException(502, "Model did not return a JSON object") from None\n    output_tokens = output.shape[1] - input_tokens\n    return {\n        "id": f"amd-{secrets.token_hex(8)}",\n        "object": "chat.completion",\n        "created": int(time.time()),\n        "model": MODEL,\n        "choices": [\n            {\n                "index": 0,\n                "message": {"role": "assistant", "content": content},\n                "finish_reason": "stop"\n                if output_tokens < request.max_tokens\n                else "length",\n            }\n        ],\n        "usage": {\n            "prompt_tokens": input_tokens,\n            "completion_tokens": output_tokens,\n            "total_tokens": input_tokens + output_tokens,\n        },\n    }\n\n\n@app.post("/v1/chat/completions", dependencies=[Depends(authorize)])\nasync def completions(request: CompletionRequest):\n    if request.model != MODEL or request.stream:\n        raise HTTPException(400, "Unsupported model or streaming request")\n    if state["active"]:\n        raise HTTPException(429, "AMD server at capacity")\n    state["active"] = True\n    task = asyncio.create_task(asyncio.to_thread(generate, request))\n    task.add_done_callback(finish_generation)\n    return await asyncio.shield(task)\n\n\ndef finish_generation(task):\n    state["active"] = False\n    if not task.cancelled():\n        task.exception()\n'
(setup_dir / 'amd_model_server.py').write_text(server_source)
key_path = setup_dir / 'api-key'
if not key_path.exists():
    descriptor = os.open(key_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(descriptor, 'w') as handle:
        handle.write(secrets.token_urlsafe(48))
server_log = (setup_dir / 'server.log').open('w')
server_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'amd_model_server:app', '--host', '127.0.0.1', '--port', '8180', '--no-access-log'], cwd=setup_dir, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
print('Model server starting; weights download may take a few minutes. PID:', server_process.pid)


In [ ]:
import urllib.request, urllib.error
api_key = key_path.read_text().strip()
try:
    request = urllib.request.Request('http://127.0.0.1:8180/health', headers={'Authorization': 'Bearer ' + api_key})
    with urllib.request.urlopen(request, timeout=10) as response:
        print(json.load(response))
except (OSError, urllib.error.URLError):
    print('Model server is not ready yet. Process exit code:', server_process.poll())
    print((setup_dir / 'server.log').read_text()[-2000:])


## Endpoint access

The platform tunnel creates a public HTTPS address. The server enforces bearer authentication on every route. Enabling this step makes that authenticated service reachable from your laptop; it does not expose Jupyter or the workspace. Run only after approving public exposure.


In [ ]:
EXPOSE_ENDPOINT = False
if EXPOSE_ENDPOINT:
    subprocess.run(['/var/run/secrets/frp-self-service/install'], check=True)
    tunnel = Path.home() / '.local/bin/rc-tunnel'
    result = subprocess.run([str(tunnel), 'expose', '--port', '8180'], capture_output=True, text=True, check=True)
    print(result.stdout)
else:
    print('Public exposure disabled. Review the server health, then explicitly enable this step.')


After the tunnel prints its HTTPS address, enter that address in the next cell. Download the resulting connection file to your laptop and merge its values into the ignored `.env.local`. Keep this file out of chat and Git. Leave AMD_ENABLED=false until `ops/check_amd.py` passes.


In [ ]:
from urllib.parse import urlsplit
from IPython.display import FileLink, display
AMD_ORIGIN = ''
address = urlsplit(AMD_ORIGIN)
if address.scheme != 'https' or not address.hostname or not address.hostname.endswith('.radeon.firstdg.ai') or address.username or address.query or address.fragment:
    raise ValueError('Set AMD_ORIGIN to the HTTPS address printed by the platform tunnel')
connection_path = Path.cwd() / 'faultline-amd-connection.env'
connection_path.write_text('AMD_ENABLED=false\nAMD_MODEL=Qwen/Qwen3-4B\nAMD_CAPACITY=1\nAMD_BASE_URL=' + AMD_ORIGIN.rstrip('/') + '/v1\nAMD_API_KEY=' + api_key + '\n')
connection_path.chmod(0o600)
display(FileLink(connection_path.name))
